In [ ]:
# Colab setup
import sys
if "google.colab" in sys.modules:
    %pip install -q "pyedautils>=0.0.32" influxdb-toolkit ephem statsmodels plotly ipywidgets

import plotly.io as pio
pio.renderers.default = "notebook_connected"

# InfluxDB — Time Series Database

[InfluxDB](https://www.influxdata.com/) is an open-source **time series database** specifically optimized for storing, querying, and analyzing time-stamped data. It is widely used in monitoring, IoT, and energy data applications.

There are two major versions:

- **InfluxDB v1.x** — uses InfluxQL (an SQL-like query language) and organizes data into databases and retention policies.
- **InfluxDB v2.x** — introduces Flux as the primary query language and organizes data into buckets and organizations.

### Why use InfluxDB for energy data?

- **High write throughput** — designed to ingest millions of data points per second, which is ideal for sensor and meter data.
- **Built-in downsampling** — aggregate high-resolution data into lower-resolution summaries automatically.
- **Retention policies** — automatically expire old data after a configurable period, keeping storage costs manageable.

```{note}
This is a conceptual and reference chapter. The connection and query examples use placeholder credentials and are shown as code snippets — running them requires access to a live InfluxDB instance.
```

## Installation

`influxdb-toolkit` is a unified Python API for **InfluxDB v1 and v2** (with extension
points for v3). It wraps the official clients, so the same `get_timeseries(...)` call
works against either server version. Install it from PyPI:

```
pip install influxdb-toolkit
```

In [ ]:
from influxdb_toolkit import InfluxDBClientFactory

print("influxdb-toolkit is ready")

## Connecting and querying (InfluxDB v2)

`InfluxDBClientFactory.get_client()` returns the right client for your server from a
**config dict**. Use it as a context manager — the connection opens on entry and closes
on exit. InfluxDB v2 authenticates with a **token** and organises data into an **org**
and a **bucket**. `get_timeseries(...)` returns a pandas DataFrame; the time range is
given as `datetime` objects, and `interval` + `aggregation` downsample on the server.

```python
from datetime import datetime, timedelta, timezone
from influxdb_toolkit import InfluxDBClientFactory

config = {
    "url": "http://localhost:8086",   # your InfluxDB URL
    "token": "my-token",              # v2 API token
    "org": "my-org",
    "bucket": "my-bucket",
}

with InfluxDBClientFactory.get_client(version=2, config=config) as client:
    df = client.get_timeseries(
        measurement="temperature",
        fields=["value"],
        start=datetime.now(timezone.utc) - timedelta(days=30),
        end=datetime.now(timezone.utc),
        interval="1h",            # server-side downsampling
        aggregation="mean",
        tags={"location": "office"},
    )
    print(df.head())
```

## InfluxDB v1

The same call works against an InfluxDB **v1** server — only the config changes: v1 uses
**username/password** and a **database** (with optional `ssl`). Pass `version=1`, or let
the factory auto-detect the version from the config keys.

```python
config_v1 = {
    "host": "localhost",
    "port": 8086,
    "username": "my-user",
    "password": "my-password",
    "database": "my-database",
    "ssl": False,
}

with InfluxDBClientFactory.get_client(version=1, config=config_v1) as client:
    df = client.get_timeseries(
        measurement="temperature",
        fields=["value"],
        start=datetime.now(timezone.utc) - timedelta(days=30),
        end=datetime.now(timezone.utc),
        interval="1h",
        aggregation="mean",
    )
    print(df.head())
```

## Query parameters

`get_timeseries(...)` accepts:

| Parameter     | Description                                                       |
|---------------|-------------------------------------------------------------------|
| `measurement` | Measurement name (like a table), e.g. `temperature`               |
| `fields`      | List of fields to return, e.g. `["value"]`                        |
| `start`/`end` | Time range as `datetime` objects                                  |
| `interval`    | Server-side downsampling window, e.g. `"5m"`, `"1h"` (optional)   |
| `aggregation` | Aggregate per interval: `"mean"`, `"sum"`, `"max"`, … (optional)  |
| `tags`        | Tag filter, e.g. `{"location": "office"}` (optional)              |
| `timezone`    | Output timezone, default `"UTC"` (optional)                       |

Other useful client methods: `list_measurements()`, `get_fields(measurement)`,
`get_tags(measurement)`, `get_schema()`, and `write_dataframe(df, measurement, …)`.

## Credentials, environment & profiles

Never hard-code credentials in notebooks. `influxdb-toolkit` reads them from
**environment variables** (or a **`.env`** file via `load_env()`), and named
**profiles** let you switch between servers without changing code.

```python
import os
from influxdb_toolkit import (
    InfluxDBClientFactory, load_env, list_profile_names, resolve_profile,
)

load_env()                    # load credentials from a local .env file, if present
print(list_profile_names())   # named profiles defined via environment variables

# Either resolve a named profile …
version, config = resolve_profile("my-server")
client = InfluxDBClientFactory.get_client(version=version, config=config)

# … or build the config explicitly from environment variables:
config = {
    "url": os.environ["INFLUXDB_URL"],
    "token": os.environ["INFLUXDB_TOKEN"],
    "org": os.environ["INFLUXDB_ORG"],
    "bucket": os.environ["INFLUXDB_BUCKET"],
}
```

## Data Schema

InfluxDB organizes data using a specific schema model. Understanding these concepts is essential for designing efficient time series storage.

| Concept         | Description                                                                 |
|-----------------|-----------------------------------------------------------------------------|
| **Measurement** | The name of the data series (similar to a table in SQL), e.g. `temperature` |
| **Tags**        | Indexed key-value pairs used for filtering and grouping, e.g. `location=office` |
| **Fields**      | The actual data values (not indexed), e.g. `value=21.5`                     |
| **Timestamp**   | The time associated with each data point (nanosecond precision)             |

A single data point in InfluxDB line protocol looks like this:

```
temperature,location=office,sensor=s1 value=21.5 1609459200000000000
```

Where:
- `temperature` is the **measurement**
- `location=office,sensor=s1` are **tags**
- `value=21.5` is a **field**
- `1609459200000000000` is the **timestamp** (Unix nanoseconds)

```{tip}
Choose tags carefully. Tags are indexed and should be used for metadata you want to filter or group by (e.g. building, floor, sensor ID). Fields hold the actual measured values and are not indexed.
```